In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
#(a) cardinality
df = pd.read_csv("retail.csv")
df["StockCode"] = df["StockCode"].astype(str).astype("category")
df["CustomerID"] = df["CustomerID"].astype(int).astype(str).astype("category")
print(df[["StockCode", "CustomerID"]].nunique())
print("one-hot columns needed:", df["StockCode"].nunique() + df["CustomerID"].nunique())
print(df["Quantity"].describe())

StockCode     3665
CustomerID    4339
dtype: int64
one-hot columns needed: 8004
count    397924.000000
mean         13.021823
std         180.420210
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64


In [3]:
#split data: 70/15/15, target: log1p(Quantity)
cols = ["StockCode", "CustomerID"]
tr, tmp = train_test_split(df, test_size=0.30, random_state=0)
va, te = train_test_split(tmp, test_size=0.50, random_state=0)
y_tr, y_va, y_te = [np.log1p(d["Quantity"].to_numpy(np.float32)) for d in (tr, va, te)]
print(len(tr), len(va), len(te))

#baseline: train mean
base = y_tr.mean()
results = {"mean baseline": [np.sqrt(np.mean((y_te - base) ** 2)), np.mean(np.abs(y_te - base))]}

278546 59689 59689


In [4]:
#train / eval helpers (MSE on log1p scale, early stopping on validation)
class Net(nn.Module):
    def __init__(self, n_dense=0, n_emb=(), emb_dim=16, hidden=128):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, emb_dim) for n in n_emb])
        d = n_dense + emb_dim * len(n_emb)
        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, dense, cats):
        parts = [dense] + [e(cats[:, i]) for i, e in enumerate(self.embs)]
        return self.mlp(torch.cat(parts, dim=1)).squeeze(-1)

def to_t(dense, cats, y):
    return (torch.tensor(dense, dtype=torch.float32).to(device), torch.tensor(cats, dtype=torch.long).to(device), torch.tensor(y).to(device))

def predict(model, data, bs=8192):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(data[0][i:i + bs], data[1][i:i + bs]) for i in range(0, len(data[2]), bs)]).cpu().numpy()

def fit(model, train, val, epochs=60, lr=1e-3, bs=512, patience=5):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(train[2])
    best, best_state, bad = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            nn.functional.mse_loss(model(train[0][idx], train[1][idx]), train[2][idx]).backward()
            opt.step()
        v = np.mean((predict(model, val) - val[2].cpu().numpy()) ** 2)
        if v < best:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    print(f"epochs run: {epoch + 1}, best val MSE: {best:.4f}")
    return model

def report(name, model, test):
    r = predict(model, test) - y_te
    results[name] = [np.sqrt(np.mean(r ** 2)), np.mean(np.abs(r))]
    return r

In [5]:
#(b) target encoding + MLP
#cross-fitted on train to avoid label leakage
enc = TargetEncoder(target_type="continuous", cv=5, random_state=0)
E_tr = enc.fit_transform(tr[cols].astype(str), y_tr).astype(np.float32)
E_va, E_te = [enc.transform(d[cols].astype(str)).astype(np.float32) for d in (va, te)]
mu_e, sd_e = E_tr.mean(0), E_tr.std(0)
E_tr, E_va, E_te = [(e - mu_e) / sd_e for e in (E_tr, E_va, E_te)]
none = lambda n: np.zeros((n, 1), dtype=np.int64)

m_te = fit(Net(n_dense=2), to_t(E_tr, none(len(tr)), y_tr), to_t(E_va, none(len(va)), y_va))
r_te = report("target encoding", m_te, to_t(E_te, none(len(te)), y_te))

epochs run: 16, best val MSE: 0.3255


In [6]:
#(c) embeddings (unseen categories -> index 0)
codes = {}
for c in cols:
    cats = tr[c].astype(str).unique()
    codes[c] = {k: i + 1 for i, k in enumerate(cats)}
def encode(d):
    return np.stack([d[c].astype(str).map(codes[c]).fillna(0).astype(np.int64).to_numpy() for c in cols], axis=1)
C_tr, C_va, C_te = encode(tr), encode(va), encode(te)
n_emb = [len(codes[c]) + 1 for c in cols]
zero = lambda n: np.zeros((n, 0), dtype=np.float32)

m_emb = fit(Net(n_emb=n_emb), to_t(zero(len(tr)), C_tr, y_tr), to_t(zero(len(va)), C_va, y_va))
r_emb = report("embeddings", m_emb, to_t(zero(len(te)), C_te, y_te))

epochs run: 25, best val MSE: 0.2896


In [7]:
#(d) target encoding + embeddings
m_both = fit(Net(n_dense=2, n_emb=n_emb), to_t(E_tr, C_tr, y_tr), to_t(E_va, C_va, y_va))
r_both = report("target enc + embeddings", m_both, to_t(E_te, C_te, y_te))
print(pd.DataFrame(results, index=["test RMSE (log1p)", "test MAE (log1p)"]).T.round(4))

epochs run: 17, best val MSE: 0.2805
                         test RMSE (log1p)  test MAE (log1p)
mean baseline                       1.0072            0.8193
target encoding                     0.5797            0.4263
embeddings                          0.5474            0.3905
target enc + embeddings             0.5369            0.3799


In [8]:
#(e) test error by whether categories were seen in training
seen = {"both seen": (C_te > 0).all(1), "unseen StockCode": C_te[:, 0] == 0, "unseen CustomerID": C_te[:, 1] == 0}
rows = {}
for gname, mask in seen.items():
    rows[gname] = [mask.sum(), np.sqrt(np.mean((y_te[mask] - base) ** 2))] + [np.sqrt(np.mean(r[mask] ** 2)) for r in (r_te, r_emb, r_both)]
print(pd.DataFrame(rows, index=["n", "mean baseline", "target encoding", "embeddings", "target enc + emb"]).T.round(4))

#embedding norm vs. training frequency for CustomerID
w = m_emb.embs[1].weight.detach().cpu().numpy()[1:]
freq = tr["CustomerID"].astype(str).map(tr["CustomerID"].astype(str).value_counts()).groupby(tr["CustomerID"].astype(str)).first()
freq = freq.reindex(list(codes["CustomerID"].keys())).to_numpy()
print("corr(log frequency, embedding norm):", np.corrcoef(np.log(freq), np.linalg.norm(w, axis=1))[0, 1].round(3))

                         n  mean baseline  target encoding  embeddings  \
both seen          59635.0          1.006           0.5779      0.5436   
unseen StockCode      30.0          1.002           0.6864      1.6751   
unseen CustomerID     24.0          2.630           2.3320      2.7498   

                   target enc + emb  
both seen                    0.5346  
unseen StockCode             0.7984  
unseen CustomerID            2.4158  
corr(log frequency, embedding norm): -0.154
